In [1]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
from utils import combine_paragraphs, read_prompt_from_file_only
from inputs import allowed_types_dict, groups_to_prompts
import json

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, MONGO_COLLECTION]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    articles_collection = db[MONGO_COLLECTION]

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'gpt-4-1106-preview', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4-turbo-preview', 'text-embedding-3-small', 'babbage-002', 'gpt-4', 'text-embedding-ada-002', 'chatgpt-4o-latest', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-4.1-nano', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'gpt-4.1-nano-2025-04-14', 'gpt-3.5-turbo-16k', 'gpt-4o-realtime-preview', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'o3-mini-2025-01-31', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4-0125-preview', 'gpt-4o-2024-11-20', 'gpt-4o-2024-05-13', 'text-embedding-3-large', 'o1-2024-12-17', 'o1', 'o1-preview', 'gpt-4-0613', 'o1-mini', 'gpt-4o-mini-tts', 'o1-pro', 'gpt-4o-trans

In [3]:
def format_nodes_for_prompt(nodes, allowed_types=None):
    """
    Format nodes into a clear ID-to-description mapping for GPT prompts.
    Falls back to `amount` for Capacity nodes if `name` is missing.
    """
    lines = ["The following is a list of known entities. You MUST ALWAYS use the ID (left value) when referring to it in a relationship:"]
    print("Allowed types in format_nodes_for_prompt: ", allowed_types)
    for node in nodes:
        node_id = node.get("id")
        node_type = node.get("type")

        name = node.get("name")
        if not name and node_type in {"Capacity", "Investment"}:
            name = node.get("amount")

        if node_id and node_type and name:
            if allowed_types is None or node_type in allowed_types:
                lines.append(f"- ID: {node_id}: Name: {name} ({node_type})")

    return "\n".join(lines)

def reconstruct_original_relationship_format(relationships_mongo):
    """
    Converts MongoDB-style stored relationships back into original model output format.
    Strips article_id and ID metadata.
    """
    simplified = []
    for rel in relationships_mongo:
        simplified.append({
            "source": rel.get("source"),
            "type": rel.get("type"),
            "target": rel.get("target")
        })
    return json.dumps({"relationships": simplified}, ensure_ascii=False)

In [4]:
articles_to_process = list(
    articles_collection.find(
        {"validation": True},
        {"_id": 1, "paragraphs": 1, "nodes": 1, "validation": 1,
         "relationships": 1}       
    )
    .sort("_id", -1)
)

In [5]:
# at least one of the following nodes is needed for the relationship type to be processed
required_node_types = {
    "ownership": ["company", "joint_venture"],
    "technological": ["product", "capacity"],
    "financial_origin": ["investment"],
    "financial_technological": ["investment"],
}

for relationship in allowed_types_dict.keys():
    print(relationship)

ownership
technological
financial_origin
financial_technological
capacities
investments


In [6]:
for relationship_group in ["ownership", "technological", "financial_origin", "financial_technological"]:

    # set parameters
    allowed_types = allowed_types_dict[relationship_group]
    output_file = f"{relationship_group}.jsonl"

    PROMPT_PATH = groups_to_prompts[relationship_group]
    system_prompt = read_prompt_from_file_only(PROMPT_PATH)

    with open(output_file, "w", encoding="utf-8") as f_out:
        for article in articles_to_process:
            articleID = str(article["_id"])

            if article.get("validation") is not True:
                print(f"⏭️ Skipping Article ID: {articleID} - not validated")
                continue

            text = combine_paragraphs(article)

            nodes = article.get("nodes")
            if not nodes:
                print(f"⚠️ Article ID: {articleID} has no nodes — skipping")
                continue

            # ✅ Correct node presence check
            required_types = required_node_types[relationship_group]
            has_required_nodes = any(node.get("type") in required_types for node in nodes)
            if not has_required_nodes:
                print(f"⏭️ Skipping Article ID: {articleID} - no required node types {required_types} found for relationship group '{relationship_group}'.")
                continue

            compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
            print("Compact nodes: ", compact_nodes)

            all_relationships = article.get("relationships")
            
            filter_relations = [entry for entry in all_relationships if entry.get('group') == relationship_group]
            print(f"{relationship_group} relations: ", filter_relations)

            user_content = f"""Here is the article text: {text}

            Here is the list of known entities:
            {compact_nodes}

            Please extract only the specified relationship types."""

            assistant_content = reconstruct_original_relationship_format(filter_relations)

            # Create fine-tuning format
            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_content
                },
                {
                    "role": "assistant",
                    "content": assistant_content
                }
            ]

            f_out.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
            print(f"✅ Wrote example for Article ID: {articleID}")

    print(f"\n🎉 Finished exporting fine-tuning examples to {output_file}")


Allowed types in format_nodes_for_prompt:  ['company', 'joint_venture', 'factory']
Compact nodes:  The following is a list of known entities. You MUST ALWAYS use the ID (left value) when referring to it in a relationship:
- ID: company_rock_tech_lithium: Name: Rock Tech Lithium (company)
- ID: company_mercedes_benz: Name: Mercedes-Benz (company)
- ID: company_automotive_cells_company: Name: Automotive Cells Company (company)
- ID: company_stellantis: Name: Stellantis (company)
- ID: company_total_energies: Name: Total Energies (company)
- ID: factory_guben_germany: Name: Guben (factory)
- ID: factory_untertuerkheim_germany: Name: Untertuerkheim (factory)
- ID: factory_sebes_romania: Name: Sebes (factory)
- ID: factory_berlin_germany: Name: Berlin (factory)
- ID: factory_kamenz_germany: Name: Kamenz (factory)
- ID: factory_jawor_poland: Name: Jawor (factory)
- ID: factory_sindelfingen_germany: Name: Sindelfingen (factory)
- ID: factory_koelleda_germany: Name: Koelleda (factory)
- ID: fa